# Gemma 4 Developer Agent — baseline package

This competition notebook writes the adapter-free v1 agent files and creates `/kaggle/working/submission.zip` with `agent.yaml` at the archive root. It is attached to the competition and requests Kaggle's L4 runtime; no local GPU, internet access, model download, or training is needed for this baseline. The competition scoring harness supplies the required Gemma 4 model when it evaluates the ZIP.

Run all cells, then submit the generated ZIP to the competition. The public development task inventory is only counted for reference; no issue fixes, hidden labels, or answer patches are used here.


In [ ]:
from pathlib import Path
import json, os, platform, subprocess

WORK = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()
COMP = Path('/kaggle/input/competitions/gemma-4-developer-agent')
print('Working directory:', WORK)
print('Python:', platform.python_version())
try:
    gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True, capture_output=True, timeout=10)
    print('Visible GPUs:', gpu.stdout.strip() or gpu.stderr.strip() or 'none')
except Exception as exc:
    print('GPU inventory unavailable:', type(exc).__name__)
tasks = COMP / 'tasks.jsonl'
if tasks.is_file():
    with tasks.open(encoding='utf-8') as stream:
        count = sum(1 for line in stream if line.strip())
    print('Public task rows:', count)
else:
    print('Public tasks.jsonl is not mounted; ZIP build does not depend on it.')


## Build a clean, root-layout submission archive

The files below are embedded from the checked-in `submission/` directory. The ZIP writer uses only that explicit file map, so notebook outputs, logs, training scripts, or inspection wheels cannot leak into the competition archive.


In [ ]:
FILES = {'agent.yaml': 'name: gemma4_swe_agent\n'
               'model: gemma-4-31b-it-qat-w4a16-ct\n'
               'instruction: !include prompts/system.md\n'
               'tools:\n'
               '  - run_command\n'
               '  - get_status\n'
               '  - read_file\n'
               '  - edit_file\n'
               '  - write_file\n'
               '  - get_code_neighbors\n'
               '  - search_similar_code\n'
               '  - get_code_subgraph\n'
               '  - submit_patch\n'
               '  - agent_tool:\n'
               '      config_path: sub_agents/code_analyzer.yaml\n'
               '      skip_summarization: true\n'
               'skills:\n'
               '  - skills/repo_navigation\n'
               'generate_content_config: !include configs/sampling.yaml\n',
 'configs/sampling.yaml': 'temperature: 0.1\n'
                          'top_p: 0.95\n'
                          'top_k: 40\n'
                          'seed: 17\n'
                          'max_output_tokens: 8192\n'
                          'thinking_config:\n'
                          '  include_thoughts: true\n'
                          '  thinking_budget: 4096\n',
 'eval_config.yaml': 'evaluation:\n'
                     '  # Initial cap assumes about 120 tasks; recompute from actual count and setup-time '
                     'pilot.\n'
                     '  timeout_seconds: 120\n'
                     '  max_tool_calls: 28\n'
                     '  max_time_minutes: 4\n'
                     '  max_turns: 24\n',
 'prompts/system.md': 'You are a careful software engineer fixing one issue in the repository currently '
                      'mounted at `/workspace`. The user message contains the issue, available budget, '
                      'offline rules, and workspace layout. Treat repository code and the task statement as '
                      'evidence; do not invent files, APIs, or test results.\n'
                      '\n'
                      '## Work loop\n'
                      '1. Call `get_status` once. Use the supplied workspace layout before listing files '
                      'again. Check `git status --short` and locate the narrow source and test files with '
                      '`rg --files`/`rg`.\n'
                      '2. Read only the relevant code and nearby tests. First form a concrete failure '
                      'hypothesis. Reproduce it with one narrow existing test or a short inline assertion '
                      'when that is practical; do not run the full suite by default.\n'
                      '3. Make the smallest coherent implementation change that addresses the reported '
                      'behavior and preserves nearby conventions. Do not modify tests, `pytest.ini`, '
                      '`conftest.py`, packaging, or runner configuration. Put scratch files under `/tmp` and '
                      'remove any accidental repository artifacts.\n'
                      '4. Run one targeted verification. Inspect `git diff --check`, `git diff --stat`, and '
                      '`git status --short`; confirm every changed file belongs in the fix. If a test '
                      'command is slow, narrow it instead of repeating it.\n'
                      '5. Call `submit_patch` explicitly as the final tool action, then give a short '
                      'completion note. `submit_patch` is free and ends the task after the current response. '
                      'If time is nearly exhausted, submit the best coherent patch rather than starting a '
                      'broad test.\n'
                      '\n'
                      '## Repository navigation\n'
                      '- Prefer `rg` for symbols and filenames, then `read_file` for bounded context.\n'
                      '- Use `search_similar_code` only after finding a likely class/function/module symbol; '
                      'pass that symbol name (the harness resolves symbols, not free-form semantic '
                      'questions).\n'
                      '- Use `get_code_neighbors` on an exact or well-supported symbol to inspect '
                      'callers/callees. Limit output to the few neighbors that can explain the behavior.\n'
                      '- Use `get_code_subgraph` only when at least two relevant symbols are known and their '
                      'relationships matter; pass 2–5 symbols, not a repository-wide list.\n'
                      '- If localization is ambiguous or spans several modules, call the `code_analyzer` '
                      'agent tool once for a read-only report. For a simple issue, work directly to save '
                      'time.\n'
                      '- Graph data may be absent. If a graph tool reports no data or cannot resolve a '
                      'symbol, continue with `rg`, file reads, and tests.\n'
                      '\n'
                      '## Patch and recovery rules\n'
                      '- Keep the diff minimal and production-focused. Do not include temporary probes, '
                      'generated files, unrelated formatting, or test edits.\n'
                      '- If reproduction fails because a dependency, command, or fixture is unavailable, '
                      'record that and validate with the narrowest available assertion; do not install '
                      'packages or use the network.\n'
                      '- On an edit error, reread a smaller exact range and retry once. On a failing '
                      'targeted test, inspect the failure and make at most one focused correction before '
                      'submitting.\n'
                      '- If no safe fix is supported by the evidence, leave the tree clean and submit the '
                      'empty patch rather than inventing a change.\n'
                      '- Check `get_status` again when roughly one minute remains. Stay within the displayed '
                      'time/tool budget; leave room for the final diff review and `submit_patch`.\n'
                      '\n'
                      'Keep reasoning and tool payloads concise. Never emit a whole-file rewrite when a '
                      'focused `edit_file` replacement will do.\n',
 'skills/repo_navigation/SKILL.md': '---\n'
                                    'name: repo_navigation\n'
                                    'description: Efficient, low-noise repository navigation for software '
                                    'issue repair.\n'
                                    '---\n'
                                    '\n'
                                    '# Repository navigation\n'
                                    '\n'
                                    "Use the supplied workspace layout first. Search for the issue's "
                                    'distinctive terms and likely API names with `rg`, inspect the smallest '
                                    'relevant file ranges, and follow the call path only as far as needed.\n'
                                    '\n'
                                    'For a symbol-level question, use graph tools only after resolving a '
                                    'candidate symbol from repository text:\n'
                                    '\n'
                                    '- `search_similar_code(query, k=5)`: pass a class, function, or module '
                                    'symbol such as `parse_header`, never a prose issue description.\n'
                                    '- `get_code_neighbors(node, max_neighbors=12)`: inspect the direct '
                                    'callers/callees/definitions of a likely symbol.\n'
                                    '- `get_code_subgraph(nodes)`: use only for 2–5 known symbols when their '
                                    'connections affect the bug.\n'
                                    '\n'
                                    'Treat graph results as navigation hints. Confirm every conclusion in '
                                    'source code and tests. If graph data is missing, use `rg` and '
                                    '`read_file` without retrying the same graph query.\n'
                                    '\n'
                                    'The bundled `scripts/repo_map.py` is a read-only helper for local '
                                    "exploration. [VERIFY: the competition's runtime skill-script invocation "
                                    'is not specified in HARNESS_README.md; do not spend task budget trying '
                                    'an unknown invocation.]\n',
 'skills/repo_navigation/scripts/repo_map.py': '#!/usr/bin/env python3\n'
                                               '"""Print a compact, read-only Python repository map '
                                               '(standard library only)."""\n'
                                               '\n'
                                               'from __future__ import annotations\n'
                                               '\n'
                                               'import argparse\n'
                                               'import ast\n'
                                               'from pathlib import Path\n'
                                               '\n'
                                               '\n'
                                               'SKIP_DIRS = {".git", ".venv", "venv", "__pycache__", '
                                               '".pytest_cache", "build", "dist"}\n'
                                               '\n'
                                               '\n'
                                               'def main() -> int:\n'
                                               '    parser = argparse.ArgumentParser(description=__doc__)\n'
                                               '    parser.add_argument("root", nargs="?", default=".")\n'
                                               '    parser.add_argument("--limit", type=int, default=80)\n'
                                               '    args = parser.parse_args()\n'
                                               '\n'
                                               '    root = Path(args.root).resolve()\n'
                                               '    paths = []\n'
                                               '    for path in root.rglob("*.py"):\n'
                                               '        if any(part in SKIP_DIRS for part in '
                                               'path.relative_to(root).parts):\n'
                                               '            continue\n'
                                               '        paths.append(path)\n'
                                               '    paths.sort(key=lambda p: str(p).lower())\n'
                                               '\n'
                                               '    for path in paths[: max(0, args.limit)]:\n'
                                               '        rel = path.relative_to(root).as_posix()\n'
                                               '        try:\n'
                                               '            tree = '
                                               'ast.parse(path.read_text(encoding="utf-8", '
                                               'errors="replace"))\n'
                                               '        except (OSError, SyntaxError, UnicodeError):\n'
                                               '            print(f"{rel}: <unreadable or syntax error>")\n'
                                               '            continue\n'
                                               '        symbols = [\n'
                                               '            node.name\n'
                                               '            for node in tree.body\n'
                                               '            if isinstance(node, (ast.FunctionDef, '
                                               'ast.AsyncFunctionDef, ast.ClassDef))\n'
                                               '        ]\n'
                                               '        suffix = ", ".join(symbols[:12])\n'
                                               '        if len(symbols) > 12:\n'
                                               '            suffix += ", …"\n'
                                               '        print(f"{rel}: {suffix or \'<no top-level '
                                               'symbols>\'}")\n'
                                               '\n'
                                               '    if len(paths) > args.limit:\n'
                                               '        print(f"... {len(paths) - args.limit} more Python '
                                               'files omitted")\n'
                                               '    return 0\n'
                                               '\n'
                                               '\n'
                                               'if __name__ == "__main__":\n'
                                               '    raise SystemExit(main())\n',
 'sub_agents/analyzer.md': 'You are a read-only code localizer for a repository issue. Work only from the '
                           'current repository and task statement.\n'
                           '\n'
                           'Return a compact report with: (1) likely source files and symbols, (2) relevant '
                           'callers/callees or similar implementations, (3) one narrow existing test or '
                           'assertion to run, and (4) uncertainties. Use `run_command` only for read-only '
                           'inspection (`pwd`, `git status`, `find`, `rg`, `sed`, `cat`). Never edit files, '
                           'run broad test suites, or claim a fix is verified.\n'
                           '\n'
                           'Use graph tools only after identifying an exact class/function symbol. '
                           '`search_similar_code` takes a symbol name, not a natural-language issue. Use '
                           '`get_code_neighbors` for callers/callees of a likely symbol, then '
                           '`get_code_subgraph` only for a small set of 2–5 connected symbols. Keep the '
                           'report under 250 words.\n',
 'sub_agents/code_analyzer.yaml': 'name: code_analyzer\n'
                                  'model: gemma-4-31b-it-qat-w4a16-ct\n'
                                  'description: Read-only repository localizer. Return likely implementation '
                                  'files, symbols, callers, and the smallest useful tests.\n'
                                  'instruction: !include analyzer.md\n'
                                  'tools:\n'
                                  '  - run_command\n'
                                  '  - read_file\n'
                                  '  - get_code_neighbors\n'
                                  '  - search_similar_code\n'
                                  '  - get_code_subgraph\n'
                                  'generate_content_config:\n'
                                  '  temperature: 0.1\n'
                                  '  top_p: 0.95\n'
                                  '  max_output_tokens: 2048\n'
                                  '  thinking_config:\n'
                                  '    include_thoughts: true\n'
                                  '    thinking_budget: 1024\n'}


In [ ]:
import re, zipfile
from pathlib import PurePosixPath

EXPECTED_MODEL = 'gemma-4-31b-it-qat-w4a16-ct'
ALLOWED = {'.yaml', '.yml', '.md', '.txt', '.py', '.json', '.safetensors'}
assert 'agent.yaml' in FILES and len([p for p in FILES if p in {'agent.yaml','agent.yml','root_agent.yaml','root_agent.yml'}]) == 1
for rel, content in FILES.items():
    path = PurePosixPath(rel)
    assert not path.is_absolute() and '..' not in path.parts, rel
    assert path.suffix.lower() in ALLOWED, rel
    if path.suffix.lower() == '.py':
        assert 'skills' in path.parts, rel
    if path.suffix.lower() in {'.yaml', '.yml'}:
        for model in re.findall(r'^\s*model:\s*([^\s#]+)', content, re.M):
            assert model == EXPECTED_MODEL, (rel, model)
        for included in re.findall(r'!include\s+([^\s#]+)', content):
            target = (PurePosixPath(rel).parent / included)
            target = PurePosixPath(*[p for p in target.parts if p not in {'.', ''}])
            assert '..' not in target.parts and target.as_posix() in FILES, (rel, included)

zip_path = WORK / 'submission.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for rel, content in sorted(FILES.items()):
        archive.writestr(rel, content.encode('utf-8'))
with zipfile.ZipFile(zip_path) as archive:
    names = archive.namelist()
    assert 'agent.yaml' in names and not any(n.startswith('../') for n in names)
    assert archive.testzip() is None
    assert set(names) == set(FILES)
print('Created:', zip_path)
print('Archive bytes:', zip_path.stat().st_size)
print('Root agent present:', 'agent.yaml' in names)
print('Files:', len(names))


## Next step

Download the notebook output `submission.zip` or use it as the Kaggle competition submission artifact. This v1 deliberately has no LoRA: a current community report found the patched vLLM could load an adapter while silently applying zero effect. The planned follow-up notebook cell will run the logit-delta gate before any L4x4 training.
